# EnvGen-LLM Getting Started

This notebook walks you through the EnvGen-LLM framework step by step.

**Prerequisites:**
- Python 3.9+
- Install: `pip install -e .` (from the project root)
- For PyTorch agents: `pip install torch numpy`
- For LLM integration: `pip install openai` or use the built-in mock LLM

## 1. Basic Training Loop

Let's start with the simplest training loop using the mock LLM and a basic PPO agent.

In [ ]:
import sys
sys.path.insert(0, '../..')

from envgen.environments.base import EnvironmentConfig
from envgen.environments.puzzle_env import PuzzleEnvironment
from envgen.agents.ppo_agent import PPOAgent
from envgen.evaluators.skill_evaluator import SkillEvaluator
from envgen.utils.types import DifficultyLevel, Episode, Transition

In [ ]:
# Create environment
config = EnvironmentConfig(
    name="tutorial_env",
    description="Tutorial puzzle environment",
    difficulty=DifficultyLevel.EASY,
    parameters={
        "grid_size": 5,
        "num_items": 2,
        "num_obstacles": 2,
        "complexity": 0.2,
        "noise_level": 0.05,
    },
    max_steps=50,
)
env = PuzzleEnvironment(config)

# Create agent
agent = PPOAgent(name="tutorial_agent", config={"learning_rate": 0.1, "initial_epsilon": 1.0})

print(f"Environment: {config.name}")
print(f"Grid size: {config.parameters['grid_size']}")
print(f"Actions available: {len(env.get_available_actions())}")

In [ ]:
# Train for 20 episodes
episodes = []
for ep_idx in range(20):
    obs = env.reset()
    transitions = []
    
    while not env.done:
        available = env.get_available_actions()
        action = agent.select_action(obs, available)
        next_obs, reward, done, info = env.step(action)
        
        transition = Transition(
            observation=obs,
            action=action,
            reward=reward,
            next_observation=next_obs,
            done=done,
        )
        transitions.append(transition)
        obs = next_obs
    
    episode = Episode(transitions=transitions, total_reward=sum(t.reward for t in transitions))
    episodes.append(episode)
    agent.train([episode])

# Evaluate
evaluator = SkillEvaluator()
result = evaluator.evaluate(episodes, agent_id="tutorial_agent")

print(f"Mean reward: {sum(e.total_reward for e in episodes) / len(episodes):.3f}")
print(f"Overall skill score: {result.overall_score:.3f}")
for ss in result.skill_scores:
    print(f"  {ss.skill_name}: {ss.score:.3f}")

## 2. Using Different Environments

EnvGen-LLM supports multiple environment types.

In [ ]:
from envgen.environments.maze_env import MazeEnvironment
from envgen.environments.resource_env import ResourceEnvironment

# Maze environment
maze_config = EnvironmentConfig(
    name="maze_tutorial",
    description="Tutorial maze",
    difficulty=DifficultyLevel.EASY,
    parameters={"maze_width": 7, "maze_height": 7, "complexity": 0.5},
    max_steps=50,
)
maze_env = MazeEnvironment(maze_config)
obs = maze_env.reset()
print(maze_env.render())

## 3. LLM-Generated Environments

The framework's core feature: an LLM generates training environments.

In [ ]:
from envgen.llm.generator import MockLLMClient, EnvironmentGenerator

llm = MockLLMClient()
generator = EnvironmentGenerator(llm)

# Generate environments
env_configs = generator.generate_initial(
    task_description="Grid navigation with item collection",
    skills=["navigation", "planning"],
    difficulty=DifficultyLevel.EASY,
    num_environments=3,
)

for cfg in env_configs:
    print(f"  {cfg.name}: difficulty={cfg.difficulty.value}, skills={cfg.target_skills}")

## 4. Full Training Pipeline

The TrainingPipeline orchestrates everything: LLM generation, training, evaluation, and adaptation.

In [ ]:
from envgen.pipeline.trainer import TrainingPipeline, PipelineConfig

pipeline_config = PipelineConfig(
    task_description="Grid navigation and item collection",
    num_cycles=3,
    episodes_per_cycle=10,
    llm_type="mock",
    agent_type="ppo",
    env_type="puzzle",
)

pipeline = TrainingPipeline(pipeline_config)
results = pipeline.run()

print(f"Completed {results['total_cycles']} cycles")
print(f"Final evaluation: {results['final_evaluation']}")

## 5. Next Steps

- Try `python examples/custom_reward.py` to experiment with reward shaping
- Try `python examples/llm_env_generation.py` to see LLM generation in action
- Try `python examples/custom_environment.py` to learn how to create new environments
- Use `python -m envgen train --config configs/default.yaml` for CLI-based training
- Use `python -m envgen benchmark --agent ppo --episodes 10` to benchmark agents